In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from termcolor import cprint
HERE = 'c:\\Users\\harry\\Documents\\code_ws\\pb_ws\\coop_assembly\\src\\coop_assembly\\planning\\'
pyplanner_path = os.path.abspath(os.path.join(HERE, '..', '..', '..', 'external', 'pyplanners'))
sys.path.extend([
    os.path.abspath(os.path.join(HERE, '..', '..', '..', 'external', 'pddlstream')),
    pyplanner_path,
])
os.environ['PYPLANNERS_PATH'] = pyplanner_path

import pddlstream
cprint("Using pddlstream from {}".format(pddlstream.__file__), 'yellow')
import strips # pyplanners
cprint("Using strips from {}".format(strips.__file__), 'yellow')

Using pddlstream from c:\Users\harry\Documents\code_ws\pb_ws\coop_assembly\external\pddlstream\pddlstream\__init__.py
Using strips from c:\Users\harry\Documents\code_ws\pb_ws\coop_assembly\external\pyplanners\strips\__init__.py


In [3]:
import numpy as np
np.set_printoptions(precision=3)

## Load shapes

In [64]:
test_file_name = 'SP_arch.json' # | SP_arch | 1_exp_tets

In [65]:
from coop_assembly.data_structure import BarStructure
from coop_assembly.planning.parsing import get_assembly_path, load_structure
from coop_assembly.help_functions.parsing import parse_saved_structure_data
from coop_assembly.help_functions.shared_const import METER_SCALE
from coop_assembly.planning.parsing import unpack_structure
from coop_assembly.planning.visualization import set_camera

import pybullet_planning as pp

connect_pb = True
viewer = True
bar_struct, o_struct = load_structure(test_file_name, connect_pb=connect_pb, viewer=viewer)
if connect_pb:
    pp.create_plane(color=pp.WHITE)
    pp.draw_pose(pp.unit_pose())

# transform model
new_world_from_base = pp.Pose(point=(np.array([0,0,0])))
world_from_base = pp.Pose(point=bar_struct.base_centroid(METER_SCALE))
# rotation = Pose(euler=Euler(roll=BASE_ROLL, pitch=BASE_PITCH, yaw=BASE_YAW))
rotation = pp.unit_pose()
tf = pp.multiply(new_world_from_base, rotation, pp.invert(world_from_base))
bar_struct.transform(tf, scale=METER_SCALE)
bar_struct.generate_grounded_connection();

element_from_index, grounded_elements, contact_from_connectors, connectors = \
    unpack_structure(bar_struct, chosen_bars=None, scale=METER_SCALE, color=pp.apply_alpha(pp.RED, 0.1))
if connect_pb:
    set_camera([bar_struct.base_centroid(METER_SCALE)], camera_dist=1.25, scale=1.)

all_elements = list(element_from_index.keys())

Parsed file name: c:\users\harry\documents\code_ws\pb_ws\coop_assembly\tests\test_data\SP_arch.json | write_time: 2020-12-30 12:10:48.047718 | 


In [51]:
print(bar_struct.base_centroid(METER_SCALE))

[ 0. -0.  0.]


## Check scaffolds work with conmech

In [66]:
from stiffness import conmech_model_from_bar_structure, create_stiffness_checker, evaluate_stiffness, draw_conmech_model

# model, fem_element_from_bar_id = conmech_model_from_bar_structure(bar_struct)
checker, fem_element_from_bar_id = create_stiffness_checker(bar_struct, presample_scaffolds=True)

if connect_pb:
    draw_conmech_model(checker.model, fem_element_from_bar_id, elements=None);

In [67]:
pp.remove_all_debug()

In [68]:
seq = [36, 1, 0, 2, 37, 35, 38, 32, 30, 31, 7, 6, 25, 3, 4, 5, 33, 9, 27, 26, 8, 12, 20, 23, 17, 34, 29, 11, 24, 10, 21, 22, 16, 28, 14, 19, 13, 15, 18]

Before adding the scaffold element

In [69]:
partial_elements = [seq[i] for i in range(22)]

deformation_result = evaluate_stiffness(bar_struct, partial_elements, checker=checker,
    fem_element_from_bar_id=fem_element_from_bar_id, verbose=True)

if connect_pb:
    draw_conmech_model(checker.model, fem_element_from_bar_id, elements=partial_elements);

Stiff: True | Compliance: 0.00001
Max translation deformation: 0.00352 / 0.005 = 0.70393, at node #110


In [70]:
pp.remove_all_debug()

 After adding the scaffold element

In [71]:
partial_elements = [seq[i] for i in range(22)] + [350]
print(partial_elements)

deformation_result = evaluate_stiffness(bar_struct, partial_elements, checker=checker,
    fem_element_from_bar_id=fem_element_from_bar_id, verbose=True)

if connect_pb:
    draw_conmech_model(checker.model, fem_element_from_bar_id, elements=partial_elements);

[36, 1, 0, 2, 37, 35, 38, 32, 30, 31, 7, 6, 25, 3, 4, 5, 33, 9, 27, 26, 8, 12, 350]
Stiff: True | Compliance: 0.00000
Max translation deformation: 0.00063 / 0.005 = 0.12567, at node #119


In [ ]:
pp.remove_all_debug()

## Define stream functions

In [74]:
from coop_assembly.planning.utils import compute_z_distance, check_connected, Command
from coop_assembly.planning.stiffness import test_stiffness

def get_wild_remove_element_gen_fn(bar_struct, element_from_index, grounded_elements, connectors, debug=False, use_fluent=True, \
    connectivity=True, stiffness=True, checker_info=(None,None)):
    def wild_gen_fn(robot_name, element, fluents=[]):
        next_printed = set()
        for fact in fluents:
            if fact[0] == 'assembled':
                #if fact[1] != element:
                next_printed.add(fact[1])
            else:
                raise NotImplementedError(fact[0])

        assert element in next_printed
        next_printed = next_printed - {element}
        print('---')
        print('Remove E{} - fluent printed ({}): {}'.format(element, len(next_printed), sorted(next_printed)))

        # # * partial ordering
        # if ordering and (outgoing_from_element[element] & next_printed):
        #     # outgoing elements must be removed BEFORE element
        #     cprint('Ordering fails.', 'red')
        #     return

        # * connectivity
        if connectivity and not check_connected(connectors, grounded_elements, next_printed):
            cprint('Connectivity fails.', 'red')
            return

        if stiffness and not test_stiffness(bar_struct, next_printed, checker=checker_info[0],
            fem_element_from_bar_id=checker_info[1]):
            cprint('Stiffness fails.', 'red')
            return

        if debug:
            num_outputs = 2 if use_fluent else 3
            outputs = [num_outputs*(None,)]
            yield outputs # For debugging
            return
        
        command = Command([])
        grasp = None
        outputs = [(command, grasp)]
        yield outputs
    # end wild_gen_fn
    return wild_gen_fn

In [92]:
def get_wild_add_element_gen_fn(bar_struct, element_from_index, grounded_elements, connectors, debug=False,
    use_fluent=True, connectivity=True, stiffness=True, checker_info=(None,None)):
    def wild_gen_fn(robot_name, element, fluents=[]):
        next_printed = set()
        for fact in fluents:
            if fact[0] == 'assembled':
                next_printed.add(fact[1])
            else:
                raise NotImplementedError(fact[0])

        assert element not in next_printed
        next_printed = next_printed | {element}
        print('---')
        print('Add E{} - fluent printed ({}): {}'.format(element, len(next_printed), sorted(next_printed)))

        # # * partial ordering
        # if ordering and (outgoing_from_element[element] & next_printed):
        #     # outgoing elements must be removed BEFORE element
        #     cprint('Ordering fails.', 'red')
        #     return

        # * connectivity
        if connectivity and not check_connected(connectors, grounded_elements, next_printed):
            cprint('Connectivity fails.', 'red')
            return

        if stiffness and not test_stiffness(bar_struct, next_printed, checker=checker_info[0],
            fem_element_from_bar_id=checker_info[1]):
            cprint('Stiffness fails.', 'red')
            return

        if debug:
            num_outputs = 2 if use_fluent else 3
            outputs = [num_outputs*(None,)]
            yield outputs # For debugging
            return
        
        command = Command([])
        grasp = None
        outputs = [(command, grasp)]
        yield outputs
    # end wild_gen_fn
    return wild_gen_fn

## Construct PDDL problem

In [93]:
printed=set()
removed=set()
debug = False

# constraint
stiffness = True
forward = True

In [94]:
from pddlstream.utils import read
from pddlstream.language.stream import StreamInfo, PartialInputs, WildOutput, DEBUG
from pddlstream.language.constants import And, PDDLProblem, Equal, print_plan

formu = 'forward' if forward else 'backward'
domain_pddl = read(os.path.join(HERE, 'pddl/stiffness/domain_{}.pddl'.format(formu)))
stream_pddl = read(os.path.join(HERE, 'pddl/stiffness/stream_{}.pddl'.format(formu)))

init = []
init.extend([
    ('Robot', 'r0'),
])

remaining = set(element_from_index) - removed - printed
for e in remaining:
    init.extend([
        ('Element', e),
        ('Removed', e) if forward else ('Assembled', e),
    ])

constant_map = {}
if forward:
    stream_map = {
        'sample-add': get_wild_add_element_gen_fn(bar_struct, element_from_index, grounded_elements, connectors,
            debug=debug, stiffness=stiffness)
    }
else:
    stream_map = {
        'sample-remove': get_wild_remove_element_gen_fn(bar_struct, element_from_index, grounded_elements, connectors,
            debug=debug, stiffness=stiffness)
    }

if debug:
    stream_map = DEBUG

goal_literals = []
if forward:
    goal_literals.extend(('Assembled', e) for e in remaining)
else:
    goal_literals.extend(('Removed', e) for e in remaining)
goal = And(*goal_literals)

pddlstream_problem = PDDLProblem(domain_pddl, constant_map, stream_pddl, stream_map, init, goal)

In [95]:
# print('Init:', pddlstream_problem.init)
# print('Goal:', pddlstream_problem.goal)

## Solve

In [96]:
costs = False

In [91]:
from pddlstream.algorithms.downward import set_cost_scale, parse_action
from pddlstream.algorithms.incremental import solve_incremental
from pddlstream.utils import INF

set_cost_scale(1)
success_cost = 0 if costs else INF
discrete_planner = {
    'search': 'eager', # eager | lazy
    # lazy might be faster but the solution quality might be low
    'evaluator': 'greedy',
    'heuristic': 'goal', # goal | add | ff
    #'heuristic': ['ff', get_bias_fn(element_from_index)],
    'successors': 'all', # all | random
    # tiebreaker
    #'successors': get_order_fn(element_from_index),
    # ? ordering scaffolds can come here
    # TODO: confirm that this is working correctly
}
solution = solve_incremental(pddlstream_problem, max_time=600, planner=discrete_planner,
                                success_cost=success_cost, unit_costs=not costs,
                                max_planner_time=300, debug=debug, verbose=True)

Calls: 0 | Successes: 0 | Results: 0 | Counts: Counter()
Iteration: 1 | Complexity: 0 | Calls: 0 | Evaluations: 79 | Solved: False | Cost: inf | Time: 0.000


TypeError: unsupported operand type(s) for +: 'set' and 'set'

In [86]:
from pddlstream.language.constants import print_plan, is_plan

plan, _, facts = solution
print('-'*10)
print_plan(plan)
assert is_plan(plan)

print('='*10)
place_actions = [action for action in reversed(plan) if action.name == 'remove_element']
commands = []

----------


AssertionError: 

In [60]:
from coop_assembly.planning.stiffness import plan_stiffness
# progression search
progression_sequence = plan_stiffness(bar_struct, all_elements, stiffness=True, heuristic='z', max_time=pp.INF, 
    max_backtrack=0, verbose=False)

Plan-stiffness success! Elements: 39, max BT: 0, stiffness failure: 0, Time: 3.222sec
